In [ ]:
# UTILS
import numpy as np
from collections import defaultdict
import os
import pickle
import copy
import os
import numpy as np
import pandas as pd
import json
from collections import defaultdict
from sklearn.model_selection import train_test_split
import datetime
import time
import argparse
import pickle
import re
import random
import sys

# MODEL
import datetime
import math
import numpy as np
import torch
from torch import nn
from torch.nn import Module, Parameter
import torch.nn.functional as F
from collections import defaultdict

# MAIN
import argparse
import os
import sys
import time
import datetime
import torch

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def data_input(opt):
    BASE_DIR = '/content/drive/MyDrive/TUGAS_AKHIR/dataset/PREPOCESSING/hasil_prepocessing'
    # BASE_DIR = './PREPOCESSING/hasil_prepocessing'
    dataset_name='music4all'
    freq = 10  
    topu = 8000  
    PATH = 'music4all'
    n_music = 56047

    if opt.valid_portion == 0:
        test_user_item_record = os.path.join(BASE_DIR, PATH, 'baseline_te_new_freq{}.lst'.format(freq))
        train_user_item_record = os.path.join(BASE_DIR, PATH, 'baseline_tr_freq{}.lst'.format(freq))
    else:
        test_user_item_record = os.path.join(BASE_DIR, PATH,
                                             'baseline_te_new_freq{}_partition{}.lst'.format(freq, opt.valid_portion))
        train_user_item_record = os.path.join(BASE_DIR, PATH,
                                              'baseline_tr_freq{}_partition{}.lst'.format(freq, opt.valid_portion))

    music2artist_dic_record = os.path.join(BASE_DIR, PATH,
                                           '{}_music_index2artist_freq{}_topu{}'.format(dataset_name, freq, topu))
    music2album_dic_record = os.path.join(BASE_DIR, PATH,
                                          '{}_music_index2album_freq{}_topu{}'.format(dataset_name, freq, topu))
    music2artist_dic = pickle.load(open(music2artist_dic_record, 'rb'))
    music2album_dic = pickle.load(open(music2album_dic_record, 'rb'))

    test_user_item_file = open(test_user_item_record, 'rb')
    train_user_item_file = open(train_user_item_record, 'rb')

    test_lines = test_user_item_file.readlines()
    train_lines = train_user_item_file.readlines()

    train_user, train_music, train_artist, train_album = [], [], [], []
    test_user, test_music, test_artist, test_album = [], [], [], []

    usr2music_record_dic = defaultdict(set)

    flag = 1
    train_lenth,test_lenth=0,0
    user_set=set()
    train_missed_artist, train_missed_album=0,0
    train_missed_artist_set, train_missed_album_set=set(),set()
    for line in train_lines:
        if flag:
            flag = 0
            continue
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')
        int_items_id = list(map(int, items_id))  
        int_items_id_plus = np.array(int_items_id) + 1  
        train_lenth+=len(int_items_id)
        for item in int_items_id_plus: 
            usr2music_record_dic[user_id].add(item)

        train_user.append(user_id)
        train_music.append(int_items_id_plus.tolist())

        for x in int_items_id:
            if music2artist_dic[x]==-1:
                train_missed_artist+=1
                train_missed_artist_set.add(x)
            if music2album_dic[x]==-1:
                train_missed_album+=1
                train_missed_album_set.add(x)

        train_artist.append([int(music2artist_dic[x]) + 1 for x in int_items_id])  
        train_album.append([int(music2album_dic[x]) + 1 for x in int_items_id])

    test_missed_artist, test_missed_album = 0, 0
    test_missed_artist_set, test_missed_album_set = set(), set()
    for line in test_lines:
        line = line.decode()
        user_id = int(line.split(',')[0])
        user_set.add(user_id)

        items_id = line.split(',')[1].split(':')

        int_items_id = list(map(int, items_id)) 
        int_items_id_plus = np.array(int_items_id) + 1  
        test_lenth+=len(int_items_id)
        test_user.append(user_id)
        test_music.append(int_items_id_plus.tolist())

        for x in int_items_id:
            if music2artist_dic[x]==-1:
                test_missed_artist+=1
                test_missed_artist_set.add(x)
            if music2album_dic[x]==-1:
                test_missed_album+=1
                test_missed_album_set.add(x)

        test_artist.append([int(music2artist_dic[x]) + 1 for x in int_items_id])
        test_album.append([int(music2album_dic[x]) + 1 for x in int_items_id])

    test_user_item_file.close()
    train_user_item_file.close()

    train_data = train_user, train_music, train_artist, train_album
    test_data = test_user, test_music, test_artist, test_album

    train_data = random_sampling(train_data, rate=opt.drop_portion)
    train_data = data_split(train_data, usr2music_record_dic,opt, '', is_test=False)

    test_new_data=copy.deepcopy(test_data)
    test_new_data = data_split(test_new_data, usr2music_record_dic,opt,'next-new-item', is_test=True)
    test_data = data_split(test_data, usr2music_record_dic,opt,'next-one-item', is_test=True)

    return train_data, test_data, test_new_data


def data_split(data, usr2music_record_dic, opt,mode, is_test=False):
    windowLenth = opt.windowLenth
    data_size = opt.data_size
    step = opt.slide_step 

    user, target, music_seq, artist_seq, album_seq = [], [], [], [], []

    if is_test == False:
        for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
            for i in range(windowLenth, int(len(music_slice) * data_size), step):
                user.append(user_id)
                target += [music_slice[i]]
                music_seq.append(music_slice[i - windowLenth:i])
                artist_seq.append(artist_slice[i - windowLenth:i])
                album_seq.append(album_slice[i - windowLenth:i])
    else:
        if mode == 'next-new-item':
            for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
                for i in range(windowLenth, int(len(music_slice))):
                    t = music_slice[i]
                    if t in usr2music_record_dic[user_id] or t in music_slice[i - windowLenth:i]:
                        continue
                    user.append(user_id)
                    target += [music_slice[i]]
                    music_seq.append(music_slice[i - windowLenth:i])
                    artist_seq.append(artist_slice[i - windowLenth:i])
                    album_seq.append(album_slice[i - windowLenth:i])

        elif mode == 'next-one-item':
            for user_id, music_slice, artist_slice, album_slice in zip(data[0], data[1], data[2], data[3]):
                for i in range(windowLenth, int(len(music_slice))):
                    user.append(user_id)
                    target += [music_slice[i]]
                    music_seq.append(music_slice[i - windowLenth:i])
                    artist_seq.append(artist_slice[i - windowLenth:i])
                    album_seq.append(album_slice[i - windowLenth:i])

    return user, target, music_seq, artist_seq, album_seq

def random_sampling(data, rate=0):
    if rate == 0:
        return data

    random.seed(42)

    new_usr, new_music, new_artist, new_album = [], [], [], []

    for usr, music, artist, album in zip(data[0], data[1], data[2], data[3]):
        total_num = len(music)
        index = list(range(total_num))
        new_index = set(random.sample(index, int(total_num * (1-rate))))

        new_music_slice = [music[idx] for idx in new_index]
        new_artist_slice = [artist[idx] for idx in new_index]
        new_album_slice = [album[idx] for idx in new_index]

        new_usr.append(usr)
        new_music.append(new_music_slice)
        new_artist.append(new_artist_slice)
        new_album.append(new_album_slice)

    return new_usr, new_music, new_artist, new_album


def build_graph(music_seq, artist_seq, album_seq):
    matrix=[]
    music_alias_input, artist_alias_input, album_alias_input = [], [], []
    artist_mask_input,album_mask_input= [], []

    items, len_list = [], []

    for music_slice, artist_slice, album_slice in zip(music_seq, artist_seq, album_seq):
        a = np.unique(music_slice + artist_slice + album_slice) 
        items.append(a)
        len_list.append(a.shape[0] + 1) 

    max_node = max(len_list) 

    for i in range(len(items)):
        items[i] = np.pad(items[i], (0, max_node - len_list[i]), 'constant', constant_values=(0))

    for i in range(len(items)):
        music_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in music_seq[i]])
        artist_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in artist_seq[i]])
        album_alias_input.append([np.where(items[i] == m)[0][0] + 1 for m in album_seq[i]])
        mask = []
        for m in album_seq[i]:
            if m != 0:
                mask += [1]
            else:
                mask += [0]
        album_mask_input.append(mask)

        mask = []
        for m in artist_seq[i]:
            if m != 0:
                mask += [1]
            else:
                mask += [0]
        artist_mask_input.append(mask)

        u_A = np.zeros((max_node, max_node))
        u_A[0][np.where(items[i] == music_seq[i][0])[0][0] + 1] = 1
        u_A[0][np.where(items[i] == artist_seq[i][0])[0][0] + 1] = 1
        u_A[0][np.where(items[i] == album_seq[i][0])[0][0] + 1] = 1

        pre = -1 
        for music, artist, album in zip(music_seq[i], artist_seq[i], album_seq[i]):
            if pre!= -1:
                pre_music_pos=np.where(items[i] == pre)[0][0] + 1
            now_music_pos=np.where(items[i] == music)[0][0] + 1
            artist_pos=np.where(items[i] == artist)[0][0] + 1
            album_pos=np.where(items[i] == album)[0][0] + 1

            if pre != -1:
                u_A[pre_music_pos][now_music_pos]=1
            pre = music

            if artist !=0 :
                u_A[artist_pos][now_music_pos]=1
            if album != 0:
                u_A[album_pos][now_music_pos]=1
            if artist and album:
                u_A[album_pos][artist_pos]=1
                u_A[artist_pos][album_pos]=1

        u_sum_in = np.sum(u_A, 0)
        u_sum_in[np.where(u_sum_in == 0)] = 1
        u_A_in = np.divide(u_A, u_sum_in)
        u_sum_out = np.sum(u_A, 1)
        u_sum_out[np.where(u_sum_out == 0)] = 1

        u_A_out = np.divide(u_A.transpose(), u_sum_out)
        u_A = np.concatenate([u_A_out, u_A_in]).transpose()
        matrix.append(u_A)

    return matrix, items, music_alias_input, artist_alias_input, album_alias_input, album_mask_input,artist_mask_input


class Data():
    def __init__(self, data, opt, shuffle=False):

        usr = data[0]
        target = data[1]
        music_seq = data[2]
        artist_seq = data[3]
        album_seq = data[4]
        matrix, items, music_alias_input, artist_alias_input, album_alias_input, album_mask_input,artist_mask_input = build_graph(
            music_seq, artist_seq, album_seq)

        self.usr = np.asarray(usr)
        self.target = np.asarray(target)
        self.items = np.asarray(items)
        self.music_alias_input = np.asarray(music_alias_input)
        self.artist_alias_input = np.asarray(artist_alias_input)
        self.album_alias_input = np.asarray(album_alias_input)
        self.artist_mask_input = np.asarray(artist_mask_input)
        self.album_mask_input = np.asarray(album_mask_input)
        self.matrix = np.asarray(matrix)

        self.windowLenth = opt.windowLenth
        self.shuffle = shuffle

    def generate_batch(self, batch_size):  
        length = len(self.target)
        if self.shuffle:
            shuffled_arg = np.arange(length)
            np.random.shuffle(shuffled_arg)
            self.usr = self.usr[shuffled_arg]
            self.target = self.target[shuffled_arg]
            self.items = self.items[shuffled_arg]
            self.matrix = self.matrix[shuffled_arg]

            self.music_alias_input = self.music_alias_input[shuffled_arg]
            self.artist_alias_input = self.artist_alias_input[shuffled_arg]
            self.artist_mask_input = self.artist_mask_input[shuffled_arg]
            self.album_alias_input = self.album_alias_input[shuffled_arg]
            self.album_mask_input = self.album_mask_input[shuffled_arg]

        n_batch = int(length / batch_size)
        if length % batch_size != 0:
            n_batch += 1
        slices = np.split(np.arange(n_batch * batch_size), n_batch)
        slices[-1] = slices[-1][:(length - batch_size * (n_batch - 1))]
        return slices

# MODEL

In [ ]:
class GNN(Module):
    def __init__(self, hidden_size, step=1):
        super(GNN, self).__init__()
        self.step = step
        self.hidden_size = hidden_size
        self.input_size = hidden_size * 2
        self.gate_size = 3 * hidden_size

        self.w_ih = Parameter(torch.Tensor(self.gate_size, self.input_size))
        self.w_hh = Parameter(torch.Tensor(self.gate_size, self.hidden_size))
        self.b_ih = Parameter(torch.Tensor(self.gate_size))
        self.b_hh = Parameter(torch.Tensor(self.gate_size))
        self.b_iah = Parameter(torch.Tensor(self.hidden_size))
        self.b_oah = Parameter(torch.Tensor(self.hidden_size))

        self.linear_edge_in = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_out = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_edge_f = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

    def GNNCell(self, A, hidden):
        input_in = torch.matmul(A[:, :, :A.shape[1]], self.linear_edge_in(hidden)) + self.b_iah
        input_out = torch.matmul(A[:, :, A.shape[1]: 2 * A.shape[1]], self.linear_edge_out(hidden)) + self.b_oah
        inputs = torch.cat([input_in, input_out], 2)
        gi = F.linear(inputs, self.w_ih, self.b_ih)
        gh = F.linear(hidden, self.w_hh, self.b_hh)
        i_r, i_i, i_n = gi.chunk(3, 2)
        h_r, h_i, h_n = gh.chunk(3, 2)
        resetgate = torch.sigmoid(i_r + h_r)
        inputgate = torch.sigmoid(i_i + h_i)
        newgate = torch.tanh(i_n + resetgate * h_n)
        hy = newgate + inputgate * (hidden - newgate)
        return hy

    def forward(self, A, hidden):
        for i in range(self.step):
            hidden = self.GNNCell(A, hidden)
        return hidden


class SessionGraph(Module):
    def __init__(self, opt, n_usr, n_album, n_artist, n_music):
        super(SessionGraph, self).__init__()
        self.hidden_size = opt.hiddenSize
        self.n_usr = n_usr
        self.n_music = n_music
        self.n_item = n_album + n_artist + n_music + 1  
        self.batch_size = opt.batchSize
        self.nonhybrid = opt.nonhybrid
        self.item_embedding = nn.Embedding(self.n_item, self.hidden_size)
        self.usr_embedding = nn.Embedding(self.n_usr, self.hidden_size)

        self.gnn = GNN(self.hidden_size, step=opt.step)

        self.linear_one_1 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_1 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_one_2 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_2 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_one_3 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)
        self.linear_two_3 = nn.Linear(self.hidden_size, self.hidden_size, bias=True)

        self.linear_transform1 = nn.Linear(self.hidden_size * 3, self.hidden_size, bias=True)
        self.linear_transform2 = nn.Linear(self.hidden_size * 3, self.hidden_size, bias=True)
        self.linear_transform3 = nn.Linear(self.hidden_size * 3, self.hidden_size, bias=True)

        self.loss_function = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=opt.lr, weight_decay=opt.l2)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt.lr_dc_step, gamma=opt.lr_dc)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.hidden_size)
        for weight in self.parameters():
            weight.data.uniform_(-stdv, stdv)

    def compute_scores(self, usr_embedding, music_embedding, artist_embedding, album_embedding):  
        ht = usr_embedding.view(usr_embedding.shape[0], usr_embedding.shape[1], 1)  

        last_music_embedding = music_embedding[:, -1, :]  
        last_artist_embedding = artist_embedding[:, -1, :]
        last_album_embedding = album_embedding[:, -1, :]

        q1_1 = self.linear_one_1(last_music_embedding).view(last_music_embedding.shape[0], 1,
                                                            last_music_embedding.shape[1])
        q1_2 = self.linear_two_1(music_embedding)

        q2_1 = self.linear_one_2(last_artist_embedding).view(last_artist_embedding.shape[0], 1,
                                                             last_artist_embedding.shape[1])
        q2_2 = self.linear_two_2(artist_embedding)

        q3_1 = self.linear_one_3(last_album_embedding).view(last_album_embedding.shape[0], 1,
                                                            last_album_embedding.shape[1])
        q3_2 = self.linear_two_3(album_embedding)

        alpha = torch.bmm((torch.tanh(q1_1 + q1_2)), ht)  
        beta = torch.bmm((torch.tanh(q2_1 + q2_2)), ht)
        delta = torch.bmm((torch.tanh(q3_1 + q3_2)), ht)

        alpha = torch.softmax(alpha, dim=1)
        beta = torch.softmax(beta, dim=1)
        delta = torch.softmax(delta, dim=1)
        a = torch.sum(alpha * music_embedding, 1)  
        b = torch.sum(beta * artist_embedding, 1)
        c = torch.sum(delta * album_embedding, 1)

        layernorm = trans_to_cuda(nn.LayerNorm(a.shape[1], eps=1e-6))
        a = layernorm(a)
        b = layernorm(b)
        c = layernorm(c)

        if not self.nonhybrid:
            st = self.linear_transform1(torch.cat([a, b, c], 1))  
            dm = self.linear_transform2(
                torch.cat([last_music_embedding, last_artist_embedding, last_album_embedding], 1))  
            all = self.linear_transform3(torch.cat([usr_embedding, st, dm], 1))

        item_embedding = self.item_embedding.weight[1:self.n_music + 1]  
        scores = torch.matmul(all, item_embedding.transpose(1, 0))

        return scores  

    def forward(self, item, usr, A):
        h1 = self.item_embedding(item)  
        h2 = self.usr_embedding(usr.view(usr.shape[0], 1))
        hidden = torch.cat([h2, h1], dim=1)
        hidden = self.gnn(A, hidden)  
        return hidden


def trans_to_cuda(variable):
    if torch.cuda.is_available():
        return variable.cuda()
    else:
        return variable


def trans_to_cpu(variable):
    if torch.cuda.is_available():
        return variable.cpu()
    else:
        return variable


def forward(model, i, data): 
    A = data.matrix[i]
    usr = data.usr[i]
    items = data.items[i]  

    music_alias_input = data.music_alias_input[i]
    artist_alias_input = data.artist_alias_input[i]
    artist_mask_input=data.artist_mask_input[i]
    album_alias_input = data.album_alias_input[i]
    album_mask_input = data.album_mask_input[i]

    items = trans_to_cuda(torch.Tensor(items).long())
    usr = trans_to_cuda(torch.Tensor(usr).long())
    A = trans_to_cuda(torch.Tensor(A).float())
    album_mask_input = trans_to_cuda(torch.Tensor(album_mask_input).long())
    artist_mask_input = trans_to_cuda(torch.Tensor(artist_mask_input).long())

    hidden = model(items, usr, A)  
    usr_embedding = hidden[:, 0, :]
    music_embedding = torch.stack(
        [hidden[i][music_alias_input[i]] for i in torch.arange(len(music_alias_input)).long()])
    artist_embedding = torch.stack(
        [hidden[i][artist_alias_input[i]] for i in torch.arange(len(artist_alias_input)).long()])
    album_embedding = torch.stack(
        [hidden[i][album_alias_input[i]] for i in torch.arange(len(album_alias_input)).long()])
    album_embedding = album_embedding * album_mask_input.view(album_mask_input.shape[0], -1, 1)
    artist_embedding = artist_embedding * artist_mask_input.view(artist_mask_input.shape[0], -1, 1)

    targets = data.target[i]
    return targets, model.compute_scores(usr_embedding, music_embedding, artist_embedding, album_embedding)

def predict(model, test_data):
    N = 21
    hit_res = np.zeros(N)
    mrr_res = np.zeros(N)
    pre_res = np.zeros(N)

    model.eval()
    hit = defaultdict(list)
    mrr = defaultdict(list)
    pre = defaultdict(list)
    slices = test_data.generate_batch(model.batch_size)
    for i in slices:
        targets, scores = forward(model, i, test_data)
        sub_scores = scores.topk(N - 1)[1]
        sub_scores = trans_to_cpu(sub_scores).detach().numpy()

        for score, target in zip(sub_scores, targets):
            target = target - 1
            for topN in range(1, N):
                topN_item = score[:topN]
                hit[topN].append(np.isin(target, topN_item))
                common = []
                if target in topN_item:
                    common.append(target)
                    mrr[topN].append(1 / (np.where(topN_item == target)[0][0] + 1))
                else:
                    mrr[topN].append(0)

                pre[topN].append(len(common) / len(topN_item))

    for topN in range(1, N):
        hit_res[topN] = np.mean(hit[topN]) * 100
        mrr_res[topN] = np.mean(mrr[topN]) * 100
        pre_res[topN] = np.mean(pre[topN]) * 100

    print("hit:\n{}".format(hit_res[1:]))
    print("mrr:\n{}".format(mrr_res[1:]))
    print("pre:\n{}".format(pre_res[1:]))

    return hit_res[1:], mrr_res[1:]

def train_test(model, train_data, test_data, test_new_data):
    print('start training: ', datetime.datetime.now())
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-one): ', datetime.datetime.now())
    hit_next_one, mrr_next_one = predict(model, test_data)

    print('start predicting (next-new): ', datetime.datetime.now())
    hit_next_new, mrr_next_new = predict(model, test_new_data)

    return (hit_next_one, mrr_next_one), (hit_next_new, mrr_next_new)

# HYPERPARAMETER TUNING

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.0/247.0 kB 24.9 MB/s eta 0:00:00


In [ ]:
import os
import sys
import time
import datetime
import json
import numpy as np
import pandas as pd
import torch
import optuna
from optuna.samplers import RandomSampler
from optuna.trial import TrialState
import copy

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Set directories
OUTPUT_DIR = "/content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def objective(trial):
    """Objective function for Optuna optimization"""
    # Get trial hyperparameters - using our recommended values
    lr = trial.suggest_categorical('lr', [0.001, 0.0005, 0.0001])
    hidden_size = trial.suggest_categorical('hidden_size', [60, 80, 100])
    l2 = trial.suggest_categorical('l2', [0.001, 0.0001, 0.00001])

    # Create a copy of default parameters to modify
    local_opt = copy.deepcopy(opt)

    # Update options being tuned
    local_opt.lr = lr
    local_opt.hiddenSize = hidden_size
    local_opt.l2 = l2

    # Fixed epoch count for all trials
    local_opt.epoch = 10

    print(f"\n\nTrial {trial.number}: lr={lr}, hidden_size={hidden_size}, l2={l2}")

    # Load data if not already loaded
    if not hasattr(objective, 'data_loaded'):
        print("Loading data...")
        objective.train_data_raw, _, objective.test_new_data_raw = data_input(local_opt)
        objective.train_data = Data(objective.train_data_raw, local_opt)
        objective.test_new_data = Data(objective.test_new_data_raw, local_opt)  # next-new items only
        objective.data_loaded = True
        print("Data loaded successfully!")

    # Initialize model with trial parameters
    n_music = 56047
    n_artist = 9348
    n_album = 21478
    n_usr = 8000
    model = trans_to_cuda(SessionGraph(local_opt, n_usr, n_album, n_artist, n_music))

    # List to store metrics for each epoch - focusing only on next-new results
    results_next_new = []

    # Train for the specified number of epochs
    for epoch in range(local_opt.epoch):
        print(f'Epoch: {epoch+1}/{local_opt.epoch}')

        # Train model and get only next-new metrics
        hit_next_new = train_test_only_new(model, objective.train_data, objective.test_new_data)

        # Store metrics for this epoch (focusing only on hit@K/recall)
        result_new = {
            'epoch': epoch + 1,
            'hit@10': float(hit_next_new[9]),
            'hit@20': float(hit_next_new[19]),
        }

        results_next_new.append(result_new)

        # Note: Pruning check removed as requested

    # Save all metrics in trial user attributes
    trial.set_user_attr('results_next_new', results_next_new)
    trial.set_user_attr('final_hit_new_10', results_next_new[-1]['hit@10'])
    trial.set_user_attr('final_hit_new_20', results_next_new[-1]['hit@20'])

    # Save trial results to JSON file
    results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
    os.makedirs(results_dir, exist_ok=True)

    trial_data = {
        "number": trial.number,
        "params": {
            "lr": lr,
            "hidden_size": hidden_size,
            "l2": l2
        },
        "hit@20": results_next_new[-1]['hit@20'],
        "hit@10": results_next_new[-1]['hit@10'],
        "epoch_results": results_next_new,
        "datetime": datetime.datetime.now().isoformat()
    }

    with open(os.path.join(results_dir, f"trial_{trial.number}.json"), "w") as f:
        json.dump(trial_data, f, indent=2, default=str)

    # Free memory
    del model
    torch.cuda.empty_cache()

    # Return metric to be optimized (hit@20 for next-new items)
    return results_next_new[-1]['hit@20']

# Modified train_test function that only evaluates next-new items
def train_test_only_new(model, train_data, test_new_data):
    print('start training: ', datetime.datetime.now())
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-new): ', datetime.datetime.now())
    hit_next_new, _ = predict(model, test_new_data)  # We only care about hit rate, not MRR

    return hit_next_new

def main():
    """Main function to run Optuna hyperparameter tuning"""
    # Ensure the model parameters are appropriate
    parser = argparse.ArgumentParser()
    parser.add_argument('--cuda_device', default='0', help='0/1/2/3')
    parser.add_argument('--output', default='display', help='local/display')
    parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
    parser.add_argument('--detail', default='optuna', help='Deskripsi kode untuk pembeda')
    parser.add_argument('--valid_portion', default=8, type=int, help='Proporsi data train-test: 1~9/0')
    parser.add_argument('--drop_portion', default=0, type=float, help='Proporsi item yang dipertahankan: 0.3/0.4/0.5')
    parser.add_argument('--data_size', default=0.1, type=float, help='Bagian data yang digunakan')
    parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
    parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
    parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
    parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
    parser.add_argument('--epoch', type=int, default=10, help='Jumlah epoch untuk training')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
    parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
    parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
    parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')
    parser.add_argument('--nonhybrid', action='store_true', help='Hanya gunakan preferensi global untuk prediksi')

    # Parse arguments with default values
    global opt
    opt = parser.parse_args(args=[])
    os.environ["CUDA_VISIBLE_DEVICES"] = opt.cuda_device

    start_time = time.time()
    print(f"Starting hyperparameter tuning with Optuna at {datetime.datetime.now()}")
    print(f"Using data_size: {opt.data_size}")
    print("Optimizing for hit@20 on next-new items only")

    # Create Optuna study with random sampler
    study = optuna.create_study(
        direction="maximize",
        sampler=RandomSampler(seed=SEED)
    )

    # Enqueue the fixed parameter trial to run first
    # Using the reference parameters: hidden_size=100, lr=0.001, l2=1e-5
    fixed_params = {
        'hidden_size': 100,
        'lr': 0.001,
        'l2': 0.00001  # 1e-5
    }
    study.enqueue_trial(fixed_params)
    print(f"Enqueued fixed parameter trial with: {fixed_params}")

    # Number of trials - running 8 random trials (including the fixed one)
    n_trials = 8

    # Callback to save trial results
    def save_trial_callback(study, trial):
        # Create directory if it doesn't exist
        results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
        os.makedirs(results_dir, exist_ok=True)

        # If this is the best trial, save it separately
        if study.best_trial.number == trial.number:
            best_trial_data = {
                "number": trial.number,
                "params": trial.params,
                "value": trial.value,
                "state": trial.state.name,
                "datetime": datetime.datetime.now().isoformat()
            }
            with open(os.path.join(results_dir, "best_trial.json"), "w") as f:
                json.dump(best_trial_data, f, indent=2, default=str)

        # Create and save summary dataframe with all completed trials
        if trial.state == TrialState.COMPLETE:
            completed_trials = [t for t in study.trials if t.state == TrialState.COMPLETE]

            trial_records = []
            for t in completed_trials:
                # Only include the 3 tuned parameters and the hit@20 metric for next-new items
                record = {
                    "trial": t.number,
                    "hit@20": t.value,  # This is hit@20 for next-new items
                    "lr": t.params.get("lr"),
                    "hidden_size": t.params.get("hidden_size"),
                    "l2": t.params.get("l2")
                }
                if 'final_hit_new_10' in t.user_attrs:
                    record["hit@10"] = t.user_attrs['final_hit_new_10']

                trial_records.append(record)

            if trial_records:
                trials_df = pd.DataFrame(trial_records)
                trials_df.to_csv(os.path.join(results_dir, "trials_summary.csv"), index=False)

    # Run optimization with callback to save results
    try:
        study.optimize(
            objective,
            n_trials=n_trials,
            callbacks=[save_trial_callback],
            timeout=None
        )

        # Print optimization results
        print("\nHyperparameter Optimization Results:")
        print(f"Best trial: {study.best_trial.number}")
        print(f"Best parameters: lr={study.best_trial.params['lr']}, "
              f"hidden_size={study.best_trial.params['hidden_size']}, "
              f"l2={study.best_trial.params['l2']}")
        print(f"Best value (Hit@20 for next-new): {study.best_trial.value:.4f}")

        # Create DataFrame with all trial results
        trials_df = pd.DataFrame([
            {
                'trial': t.number,
                'state': t.state.name,
                'lr': t.params.get('lr'),
                'hidden_size': t.params.get('hidden_size'),
                'l2': t.params.get('l2'),
                'hit@20': t.value if t.state == TrialState.COMPLETE else None
            }
            for t in study.trials
        ])

        # Save final results
        results_path = f"{OUTPUT_DIR}/final_hyperparameter_results.csv"
        trials_df.to_csv(results_path, index=False)
        print(f"\nSaved final results to: {results_path}")

    except Exception as e:
        print(f"Error during optimization: {e}")
        import traceback
        traceback.print_exc()

    end_time = time.time()
    print(f"Total tuning time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

[I 2025-06-26 01:46:30,641] A new study created in memory with name: no-name-bacb3ebc-56c5-4b37-8385-78a10563d044


Starting hyperparameter tuning with Optuna at 2025-06-26 01:46:30.641338
Using data_size: 0.1
Optimizing for hit@20 on next-new items only
Enqueued fixed parameter trial with: {'hidden_size': 100, 'lr': 0.001, 'l2': 1e-05}


Trial 0: lr=0.001, hidden_size=100, l2=1e-05
Loading data...
Data loaded successfully!
Epoch: 1/10
start training:  2025-06-26 01:47:34.210387


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[0/868] Loss: 10.9644
[174/868] Loss: 9.8911
[348/868] Loss: 7.6850
[522/868] Loss: 9.0789
[696/868] Loss: 8.8195
	Loss:	7611.766
start predicting (next-new):  2025-06-26 01:49:47.402814
hit:
[ 2.59955732  5.18112843  7.66102242  9.67731279 11.12575294 12.41378551
 13.47203512 14.42530402 15.17448455 15.84401188 16.45444168 16.98558534
 17.43744288 17.87902258 18.25893529 18.6113181  18.92626023 19.2290892
 19.50145174 19.76059993]
mrr:
[2.59955732 3.89034288 4.71697421 5.2210468  5.51073483 5.72540692
 5.87658544 5.99574405 6.07898633 6.14593906 6.20143268 6.24569465
 6.28045293 6.31199433 6.33732185 6.35934577 6.37787178 6.39469561
 6.40903048 6.42198789]
pre:
[2.59955732 2.59056422 2.55367414 2.4193282  2.22515059 2.06896425
 1.92457645 1.803163   1.68605384 1.58440119 1.49585833 1.41546545
 1.34134176 1.27707304 1.21726235 1.16320738 1.11330943 1.06828273
 1.0263922  0.98803   ]
Epoch: 2/10
start training:  2025-06-26 01:55:08.133037
[0/868] Loss: 6.8607
[174/868] Loss: 7.0813
[348

[I 2025-06-26 03:03:02,049] Trial 0 finished with value: 26.252518986458956 and parameters: {'lr': 0.001, 'hidden_size': 100, 'l2': 1e-05}. Best is trial 0 with value: 26.252518986458956.




Trial 1: lr=0.0005, hidden_size=60, l2=0.0001
Epoch: 1/10
start training:  2025-06-26 03:03:02.149580
[0/868] Loss: 10.9965
[174/868] Loss: 10.0643
[348/868] Loss: 8.6147
[522/868] Loss: 9.7103
[696/868] Loss: 9.4439
	Loss:	8146.011
start predicting (next-new):  2025-06-26 03:05:05.067842
hit:
[0.85049058 1.98472274 2.88807074 3.67836259 4.3842294  5.00640529
 5.42082215 5.9266383  6.34729528 6.71252537 7.00911423 7.2844133
 7.55530758 7.83464437 8.04827644 8.25052949 8.44544123 8.60254523
 8.74973847 8.89509637]
mrr:
[0.85049058 1.41760666 1.71872266 1.91629562 2.05746899 2.16116497
 2.22036738 2.2835944  2.33033406 2.36685707 2.39381969 2.41676128
 2.4375993  2.45755193 2.47179407 2.48443488 2.49590028 2.50462828
 2.51237529 2.51964319]
pre:
[0.85049058 0.99236137 0.96269025 0.91959065 0.87684588 0.83440088
 0.77440316 0.74082979 0.70525503 0.67125254 0.6371922  0.60703444
 0.58117751 0.55961745 0.53655176 0.51565809 0.49679066 0.47791918
 0.46051255 0.44475482]
Epoch: 2/10
start t

[I 2025-06-26 04:16:31,598] Trial 1 finished with value: 21.649151528276885 and parameters: {'lr': 0.0005, 'hidden_size': 60, 'l2': 0.0001}. Best is trial 0 with value: 26.252518986458956.




Trial 2: lr=0.0001, hidden_size=60, l2=1e-05
Epoch: 1/10
start training:  2025-06-26 04:16:31.683137
[0/868] Loss: 11.0032
[174/868] Loss: 10.7587
[348/868] Loss: 9.6051
[522/868] Loss: 10.2351
[696/868] Loss: 10.0163
	Loss:	8697.022
start predicting (next-new):  2025-06-26 04:18:39.060245
hit:
[0.11268908 0.29842419 0.51646105 0.90701866 1.01567002 1.20874643
 1.32180258 1.52295444 1.76228109 1.99279818 2.04418734 2.13595369
 2.25341463 2.3205876  2.40794917 2.47035029 2.51990412 2.56101545
 2.6241507  2.6836153 ]
mrr:
[0.11268908 0.20555664 0.27823559 0.37587499 0.39760527 0.42978467
 0.44593555 0.47107953 0.49767138 0.52072309 0.52539483 0.53304202
 0.54207748 0.54687555 0.55269965 0.55659973 0.55951466 0.56179862
 0.56512153 0.56809476]
pre:
[0.11268908 0.14921209 0.17215368 0.22675466 0.203134   0.20145774
 0.18882894 0.1903693  0.19580901 0.19927982 0.18583521 0.17799614
 0.17333959 0.16575626 0.16052994 0.15439689 0.14822965 0.14227864
 0.13811319 0.13418077]
Epoch: 2/10
start

In [ ]:
import os
import sys
import time
import datetime
import json
import numpy as np
import pandas as pd
import torch
import optuna
from optuna.samplers import RandomSampler
from optuna.trial import TrialState
import copy
import random
import argparse
from collections import defaultdict

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Variabel global untuk menyimpan semua hasil trial
all_trial_results = []

def objective(trial):
    """Objective function for Optuna optimization"""
    global all_trial_results, OUTPUT_DIR

    # Get last trial number
    last_trial_number = -1
    if all_trial_results:
        last_trial_number = max(t['number'] for t in all_trial_results)

    # Calculate corrected trial number
    corrected_trial_number = last_trial_number + 1

    # Get trial hyperparameters using random sampling
    lr = trial.suggest_categorical('lr', [0.005, 0.001, 0.0005])
    hidden_size = trial.suggest_categorical('hidden_size', [60, 80, 100])
    l2 = trial.suggest_categorical('l2', [0.001, 0.0001, 0.00001])

    # Display trial info with corrected number
    print(f"\n\nTrial {corrected_trial_number}: lr={lr}, hidden_size={hidden_size}, l2={l2}")

    # Create a copy of default parameters to modify
    local_opt = copy.deepcopy(opt)

    # Update options being tuned
    local_opt.lr = lr
    local_opt.hiddenSize = hidden_size
    local_opt.l2 = l2

    # Fixed epoch count for all trials
    local_opt.epoch = 10

    # Load data if not already loaded
    if not hasattr(objective, 'data_loaded'):
        print("Loading data...")
        objective.train_data_raw, _, objective.test_new_data_raw = data_input(local_opt)
        objective.train_data = Data(objective.train_data_raw, local_opt)
        objective.test_new_data = Data(objective.test_new_data_raw, local_opt)  # next-new items only
        objective.data_loaded = True
        print("Data loaded successfully!")

    # Initialize model with trial parameters
    n_music = 56047
    n_artist = 9348
    n_album = 21478
    n_usr = 8000
    model = trans_to_cuda(SessionGraph(local_opt, n_usr, n_album, n_artist, n_music))

    # List to store metrics for each epoch - focusing only on next-new results
    results_next_new = []

    # Train for the specified number of epochs
    for epoch in range(local_opt.epoch):
        print(f'Epoch: {epoch+1}/{local_opt.epoch}')

        # Train model and get only next-new metrics
        hit_next_new = train_test_only_new(model, objective.train_data, objective.test_new_data)

        # Store metrics for this epoch (focusing only on hit@K/recall)
        result_new = {
            'epoch': epoch + 1,
            'hit@10': float(hit_next_new[9]),
            'hit@20': float(hit_next_new[19]),
        }

        results_next_new.append(result_new)

    # Save all metrics in trial user attributes
    trial.set_user_attr('results_next_new', results_next_new)
    trial.set_user_attr('final_hit_new_10', results_next_new[-1]['hit@10'])
    trial.set_user_attr('final_hit_new_20', results_next_new[-1]['hit@20'])

    # Tambahkan hasil trial ke list global hasil
    trial_data = {
        "number": corrected_trial_number,  # Use corrected trial number
        "params": {
            "lr": lr,
            "hidden_size": hidden_size,
            "l2": l2
        },
        "hit@20": results_next_new[-1]['hit@20'],
        "hit@10": results_next_new[-1]['hit@10'],
        "epoch_results": results_next_new,
        "datetime": datetime.datetime.now().isoformat()
    }

    all_trial_results.append(trial_data)

    # Simpan semua hasil trial ke satu file JSON
    results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
    os.makedirs(results_dir, exist_ok=True)

    with open(os.path.join(results_dir, "all_trials.json"), "w") as f:
        json.dump(all_trial_results, f, indent=2, default=str)

    # Free memory
    del model
    torch.cuda.empty_cache()

    # Return metric to be optimized (hit@20 for next-new items)
    return results_next_new[-1]['hit@20']

# Modified train_test function that only evaluates next-new items
def train_test_only_new(model, train_data, test_new_data):
    print('start training: ', datetime.datetime.now())
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-new): ', datetime.datetime.now())
    hit_next_new, _ = predict(model, test_new_data)  # We only care about hit rate, not MRR

    return hit_next_new

def main():
    """Main function to run Optuna hyperparameter tuning"""
    global all_trial_results, OUTPUT_DIR  # Make OUTPUT_DIR global so objective function can access it

    # Ensure the model parameters are appropriate
    parser = argparse.ArgumentParser()
    parser.add_argument('--cuda_device', default='0', help='0/1/2/3')
    parser.add_argument('--output', default='display', help='local/display')
    parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
    parser.add_argument('--detail', default='model_optuna', help='Deskripsi kode untuk pembeda')
    parser.add_argument('--valid_portion', default=8, type=int, help='Proporsi data train-test: 1~9/0')
    parser.add_argument('--drop_portion', default=0, type=float, help='Proporsi item yang dipertahankan: 0.3/0.4/0.5')
    parser.add_argument('--data_size', default=0.1, type=float, help='Bagian data yang digunakan')
    parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
    parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
    parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
    parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
    parser.add_argument('--epoch', type=int, default=10, help='Jumlah epoch untuk training')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
    parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
    parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
    parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')
    parser.add_argument('--nonhybrid', action='store_true', help='Hanya gunakan preferensi global untuk prediksi')
    parser.add_argument('--input_json', default=None, type=str, help='Path ke file JSON dengan trial sebelumnya')

    # Parse arguments with default values
    global opt
    opt = parser.parse_args(args=[])
    os.environ["CUDA_VISIBLE_DEVICES"] = opt.cuda_device

    # Set up direktori input dan output
    global INPUT_DIR  # Make INPUT_DIR global too
    INPUT_DIR = "/content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5"
    OUTPUT_DIR = "/content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    start_time = time.time()
    print(f"Starting hyperparameter tuning with Optuna at {datetime.datetime.now()}")
    print(f"Using data_size: {opt.data_size}")
    print(f"Input directory: {INPUT_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print("Optimizing for hit@20 on next-new items only")

    # Initialize all_trial_results
    all_trial_results = []

    # Set up hasil direktori
    results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
    os.makedirs(results_dir, exist_ok=True)

    # Cek input JSON di direktori input
    input_json_path = os.path.join(INPUT_DIR, "optuna_results", "all_trials.json")

    # Muat data trial sebelumnya
    if os.path.exists(input_json_path):
        try:
            with open(input_json_path, 'r') as f:
                all_trial_results = json.load(f)
            print(f"Loaded {len(all_trial_results)} existing trials from {input_json_path}")

            # Tampilkan informasi trial yang sudah ada
            print("\nExisting trials:")
            for t in all_trial_results:
                print(f"Trial {t['number']}: lr={t['params']['lr']}, "
                      f"hidden_size={t['params']['hidden_size']}, "
                      f"l2={t['params']['l2']} → hit@20={t['hit@20']:.4f}")
            print()
        except Exception as e:
            print(f"Error loading existing trials: {e}")
            all_trial_results = []
    else:
        print("No existing trials found. Starting fresh.")
        all_trial_results = []

    # Define parameter ranges that will be used in the objective function
    parameter_ranges = {
        'lr': [0.005, 0.001, 0.0005],
        'hidden_size': [60, 80, 100],
        'l2': [0.001, 0.0001, 0.00001]
    }

    print(f"Using random sampling with Optuna to select parameters")
    print(f"Parameter ranges:")
    print(f"  lr: {parameter_ranges['lr']}")
    print(f"  hidden_size: {parameter_ranges['hidden_size']}")
    print(f"  l2: {parameter_ranges['l2']}")

    # Create Optuna study with random sampler
    study = optuna.create_study(
        direction="maximize",
        sampler=RandomSampler(seed=SEED)
    )

    # Total number of trials we want
    n_trials = 8

    # Calculate remaining trials
    remaining_trials = max(0, n_trials - len(all_trial_results))
    print(f"Will run {remaining_trials} more trials to reach total of {n_trials}")

    # Jika semua trial sudah selesai, hanya tampilkan hasil
    if remaining_trials == 0:
        print("All trials already completed. No new trials to run.")

        # Find best trial from existing results
        best_trial = max(all_trial_results, key=lambda x: x['hit@20'])
        print("\nBest Trial Results:")
        print(f"Trial {best_trial['number']}: lr={best_trial['params']['lr']}, "
              f"hidden_size={best_trial['params']['hidden_size']}, "
              f"l2={best_trial['params']['l2']}")
        print(f"Hit@20: {best_trial['hit@20']:.4f}, Hit@10: {best_trial['hit@10']:.4f}")
    else:
        # Setup callback to properly show trial numbers
        last_trial_number = -1
        if all_trial_results:
            last_trial_number = max(t['number'] for t in all_trial_results)

        def trial_callback(study, trial):
            adjusted_number = last_trial_number + 1 + trial.number
            print(f"\nRunning Trial {adjusted_number} with parameters:")
            print(f"  lr={trial.params['lr']}")
            print(f"  hidden_size={trial.params['hidden_size']}")
            print(f"  l2={trial.params['l2']}")

        # Run optimization for remaining trials
        try:
            study.optimize(
                objective,
                n_trials=remaining_trials,
                timeout=None,
                callbacks=[trial_callback]
            )

            # Print optimization results
            print("\nHyperparameter Optimization Results:")

            # Find best trial from all_trial_results
            if all_trial_results:
                best_trial = max(all_trial_results, key=lambda x: x['hit@20'])
                print(f"Best trial: {best_trial['number']}")
                print(f"Best parameters: lr={best_trial['params']['lr']}, "
                      f"hidden_size={best_trial['params']['hidden_size']}, "
                      f"l2={best_trial['params']['l2']}")
                print(f"Best value (Hit@20 for next-new): {best_trial['hit@20']:.4f}")

        except Exception as e:
            print(f"Error during optimization: {e}")
            import traceback
            traceback.print_exc()

    # Create DataFrame with all trial results
    if all_trial_results:
        trials_df = pd.DataFrame([
            {
                'trial': t['number'],
                'hit@20': t['hit@20'],
                'hit@10': t['hit@10'],
                'lr': t['params']['lr'],
                'hidden_size': t['params']['hidden_size'],
                'l2': t['params']['l2'],
                'datetime': t['datetime']
            }
            for t in all_trial_results
        ])

        # Save final results
        results_path = os.path.join(OUTPUT_DIR, "final_hyperparameter_results.csv")
        trials_df.to_csv(results_path, index=False)
        print(f"\nSaved final results to: {results_path}")

    end_time = time.time()
    print(f"Total tuning time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

[I 2025-06-26 16:11:18,101] A new study created in memory with name: no-name-3131e935-0ddb-445c-a376-182a102e3798


Starting hyperparameter tuning with Optuna at 2025-06-26 16:11:18.097923
Using data_size: 0.1
Input directory: /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5
Output directory: /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/output
Optimizing for hit@20 on next-new items only
Loaded 1 existing trials from /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/optuna_results/all_trials.json

Existing trials:
Trial 0: lr=0.001, hidden_size=100, l2=1e-05 → hit@20=26.2525

Using random sampling with Optuna to select parameters
Parameter ranges:
  lr: [0.005, 0.001, 0.0005]
  hidden_size: [60, 80, 100]
  l2: [0.001, 0.0001, 1e-05]
Will run 7 more trials to reach total of 8


Trial 1: lr=0.001, hidden_size=60, l2=0.0001
Loading data...
Data loaded successfully!
Epoch: 1/10
start training:  2025-06-26 16:12:30.173913


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


[0/868] Loss: 10.9565
[174/868] Loss: 9.9703
[348/868] Loss: 7.9509
[522/868] Loss: 9.2035
[696/868] Loss: 9.0649
	Loss:	7816.157
start predicting (next-new):  2025-06-26 16:14:45.122343
hit:
[ 1.72961227  3.81637919  5.67409729  7.21944272  8.33568867  9.46918669
 10.32775272 11.09712184 11.82317724 12.38772386 12.93575254 13.40706454
 13.80569759 14.18414204 14.52220929 14.8437586  15.14291692 15.41564653
 15.67626298 15.93834769]
mrr:
[1.72961227 2.77299573 3.3922351  3.77857145 4.00182064 4.19073698
 4.31338927 4.40956041 4.49023323 4.5466879  4.59650868 4.63578468
 4.66644877 4.69348051 4.71601833 4.73611516 4.75371271 4.76886435
 4.78258101 4.79568524]
pre:
[1.72961227 1.9081896  1.89136576 1.80486068 1.66713773 1.57819778
 1.47539325 1.38714023 1.31368636 1.23877239 1.1759775  1.11725538
 1.06197674 1.013153   0.96814729 0.92773491 0.89075982 0.85642481
 0.82506647 0.79691738]
Epoch: 2/10
start training:  2025-06-26 16:20:49.019174
[0/868] Loss: 7.5590
[174/868] Loss: 8.1144
[34

[I 2025-06-26 17:34:38,991] Trial 0 finished with value: 25.84250691000657 and parameters: {'lr': 0.001, 'hidden_size': 60, 'l2': 0.0001}. Best is trial 0 with value: 25.84250691000657.


...


Running Trial 2 with parameters:
  lr=0.0005
  hidden_size=60
  l2=1e-05


Trial 3: lr=0.0005, hidden_size=100, l2=0.0001
Epoch: 1/10
start training:  2025-06-26 18:56:35.384901
[0/868] Loss: 11.0144
[174/868] Loss: 9.9980
[348/868] Loss: 8.2434
[522/868] Loss: 9.3553
[696/868] Loss: 9.2755
	Loss:	7925.671
start predicting (next-new):  2025-06-26 18:58:51.405940
hit:
[ 1.58131784  3.23935235  4.90693056  6.30434862  7.30056418  8.24135286
  9.12524639  9.88433769 10.56230752 11.14814393 11.59266016 11.97514233
 12.38111669 12.7004636  13.01393747 13.30318503 13.56747213 13.82074727
 14.0497961  14.28765449]
mrr:
[1.58131784 2.41033509 2.9661945  3.31554901 3.51479212 3.67159024
 3.79786074 3.89274715 3.96807714 4.02666078 4.06707134 4.09894486
 4.13017365 4.15298415 4.17388241 4.19196038 4.20750668 4.22157752
 4.23363272 4.24552564]
pre:
[1.58131784 1.61967617 1.63564352 1.57608716 1.46011284 1.37355881
 1.30360663 1.23554221 1.17358972 1.11481439 1.0538782  0.99792853
 0.95239359 0.9

[I 2025-06-26 20:18:32,563] Trial 2 finished with value: 25.493427693617832 and parameters: {'lr': 0.0005, 'hidden_size': 100, 'l2': 0.0001}. Best is trial 0 with value: 25.84250691000657.



Running Trial 3 with parameters:
  lr=0.0005
  hidden_size=100
  l2=0.0001


Trial 4: lr=0.001, hidden_size=60, l2=0.0001
Epoch: 1/10
start training:  2025-06-26 20:18:32.651610
[0/868] Loss: 11.0171
[174/868] Loss: 9.9969
[348/868] Loss: 7.9625
[522/868] Loss: 9.1558
[696/868] Loss: 9.0561
	Loss:	7824.574
start predicting (next-new):  2025-06-26 20:20:46.683049
hit:
[ 1.7780649   3.84244084  5.66859131  7.11189255  8.38560957  9.46257952
 10.27562943 11.01416505 11.76775037 12.42149388 12.97723093 13.44377108
 13.83579695 14.20873542 14.58461042 14.95020758 15.23982219 15.51695659
 15.79335685 16.0539733 ]
mrr:
[1.7780649  2.81025287 3.4189697  3.77979501 4.03453841 4.2140334
 4.33018339 4.42250034 4.50623204 4.57160639 4.62212794 4.66100629
 4.69116213 4.71780059 4.74285892 4.76570874 4.7827449  4.79814125
 4.81268864 4.82571946]
pre:
[1.7780649  1.92122042 1.88953044 1.77797314 1.67712191 1.57709659
 1.46794706 1.37677063 1.30752782 1.24214939 1.17974827 1.12031426
 1.06429207 1.01

[I 2025-06-26 21:40:21,117] Trial 3 finished with value: 25.785611769585692 and parameters: {'lr': 0.001, 'hidden_size': 60, 'l2': 0.0001}. Best is trial 0 with value: 25.84250691000657.



Running Trial 4 with parameters:
  lr=0.001
  hidden_size=60
  l2=0.0001


Trial 5: lr=0.0005, hidden_size=100, l2=0.0001
Epoch: 1/10
start training:  2025-06-26 21:40:21.266079
[0/868] Loss: 10.9522
[174/868] Loss: 10.0249
[348/868] Loss: 8.3566
[522/868] Loss: 9.4876
[696/868] Loss: 9.2859
	Loss:	7933.619
start predicting (next-new):  2025-06-26 21:42:33.949474
hit:
[ 1.42274558  3.32341033  4.9755718   6.19716552  7.26716123  8.20427925
  9.08083148  9.80064677 10.40520352 10.90881728 11.37499036 11.75453601
 12.09370446 12.40497594 12.68835044 12.97429441 13.24335336 13.49479318
 13.72824679 13.95619441]
mrr:
[1.42274558 2.37307795 2.92379844 3.22919687 3.44319601 3.59938235
 3.7246041  3.81458101 3.88175398 3.93211536 3.97449473 4.00612353
 4.03221341 4.05444709 4.07333872 4.09121022 4.10703722 4.1210061
 4.13329313 4.14469051]
pre:
[1.42274558 1.66170517 1.65852393 1.54929138 1.45343225 1.36737987
 1.29726164 1.22508085 1.15613372 1.09088173 1.03409003 0.97954467
 0.93028496 0.8

In [ ]:
import os
import sys
import time
import datetime
import json
import numpy as np
import pandas as pd
import torch
import optuna
from optuna.samplers import RandomSampler
from optuna.trial import TrialState
import copy
import random
import argparse
from collections import defaultdict

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Variabel global untuk menyimpan semua hasil trial
all_trial_results = []

def objective(trial):
    """Objective function for Optuna optimization"""
    global all_trial_results, OUTPUT_DIR

    # Get trial hyperparameters using random sampling
    lr = trial.suggest_categorical('lr', [0.005, 0.001, 0.0005])
    hidden_size = trial.suggest_categorical('hidden_size', [60, 80, 100])
    l2 = trial.suggest_categorical('l2', [0.001, 0.0001, 0.00001])

    # Check if this combination has already been tried
    for t in all_trial_results:
        params = t['params']
        if (params['lr'] == lr and
            params['hidden_size'] == hidden_size and
            params['l2'] == l2):
            print(f"\nSkipping duplicate parameters: lr={lr}, hidden_size={hidden_size}, l2={l2}")
            print(f"This combination was already tried in Trial {t['number']}")
            raise optuna.exceptions.TrialPruned()

    # Get last trial number
    last_trial_number = -1
    if all_trial_results:
        last_trial_number = max(t['number'] for t in all_trial_results)

    # Calculate corrected trial number
    corrected_trial_number = last_trial_number + 1

    # Display trial info with corrected number
    print(f"\n\nTrial {corrected_trial_number}: lr={lr}, hidden_size={hidden_size}, l2={l2}")

    # Create a copy of default parameters to modify
    local_opt = copy.deepcopy(opt)

    # Update options being tuned
    local_opt.lr = lr
    local_opt.hiddenSize = hidden_size
    local_opt.l2 = l2

    # Fixed epoch count for all trials
    local_opt.epoch = 10

    # Load data if not already loaded
    if not hasattr(objective, 'data_loaded'):
        print("Loading data...")
        objective.train_data_raw, _, objective.test_new_data_raw = data_input(local_opt)
        objective.train_data = Data(objective.train_data_raw, local_opt)
        objective.test_new_data = Data(objective.test_new_data_raw, local_opt)  # next-new items only
        objective.data_loaded = True
        print("Data loaded successfully!")

    # Initialize model with trial parameters
    n_music = 56047
    n_artist = 9348
    n_album = 21478
    n_usr = 8000
    model = trans_to_cuda(SessionGraph(local_opt, n_usr, n_album, n_artist, n_music))

    # List to store metrics for each epoch - focusing only on next-new results
    results_next_new = []

    # Train for the specified number of epochs
    for epoch in range(local_opt.epoch):
        print(f'Epoch: {epoch+1}/{local_opt.epoch}')

        # Train model and get only next-new metrics
        hit_next_new = train_test_only_new(model, objective.train_data, objective.test_new_data)

        # Store metrics for this epoch (focusing only on hit@K/recall)
        result_new = {
            'epoch': epoch + 1,
            'hit@10': float(hit_next_new[9]),
            'hit@20': float(hit_next_new[19]),
        }

        results_next_new.append(result_new)

    # Save all metrics in trial user attributes
    trial.set_user_attr('results_next_new', results_next_new)
    trial.set_user_attr('final_hit_new_10', results_next_new[-1]['hit@10'])
    trial.set_user_attr('final_hit_new_20', results_next_new[-1]['hit@20'])

    # Tambahkan hasil trial ke list global hasil
    trial_data = {
        "number": corrected_trial_number,  # Use corrected trial number
        "params": {
            "lr": lr,
            "hidden_size": hidden_size,
            "l2": l2
        },
        "hit@20": results_next_new[-1]['hit@20'],
        "hit@10": results_next_new[-1]['hit@10'],
        "epoch_results": results_next_new,
        "datetime": datetime.datetime.now().isoformat()
    }

    all_trial_results.append(trial_data)

    # Simpan semua hasil trial ke satu file JSON
    results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
    os.makedirs(results_dir, exist_ok=True)

    with open(os.path.join(results_dir, "all_trials.json"), "w") as f:
        json.dump(all_trial_results, f, indent=2, default=str)

    # Free memory
    del model
    torch.cuda.empty_cache()

    # Return metric to be optimized (hit@20 for next-new items)
    return results_next_new[-1]['hit@20']

# Modified train_test function that only evaluates next-new items
def train_test_only_new(model, train_data, test_new_data):
    print('start training: ', datetime.datetime.now())
    model.scheduler.step()
    model.train()
    total_loss = 0.0
    slices = train_data.generate_batch(model.batch_size)
    for i, j in zip(slices, np.arange(len(slices))):
        model.optimizer.zero_grad()
        targets, scores = forward(model, i, train_data)
        targets = trans_to_cuda(torch.Tensor(targets).long())
        loss = model.loss_function(scores, targets-1)
        loss.backward()
        model.optimizer.step()
        total_loss += loss
        if j % int(len(slices) / 5 + 1) == 0:
            print('[%d/%d] Loss: %.4f' % (j, len(slices), loss.item()))
    print('\tLoss:\t%.3f' % total_loss)

    print('start predicting (next-new): ', datetime.datetime.now())
    hit_next_new, _ = predict(model, test_new_data)  # We only care about hit rate, not MRR

    return hit_next_new

def main():
    """Main function to run Optuna hyperparameter tuning"""
    global all_trial_results, OUTPUT_DIR  # Make OUTPUT_DIR global so objective function can access it

    # Ensure the model parameters are appropriate
    parser = argparse.ArgumentParser()
    parser.add_argument('--cuda_device', default='0', help='0/1/2/3')
    parser.add_argument('--output', default='display', help='local/display')
    parser.add_argument('--dataset', default='music4all', help='dataset name: music4all')
    parser.add_argument('--detail', default='model_optuna', help='Deskripsi kode untuk pembeda')
    parser.add_argument('--valid_portion', default=8, type=int, help='Proporsi data train-test: 1~9/0')
    parser.add_argument('--drop_portion', default=0, type=float, help='Proporsi item yang dipertahankan: 0.3/0.4/0.5')
    parser.add_argument('--data_size', default=0.1, type=float, help='Bagian data yang digunakan')
    parser.add_argument('--slide_step', default=1, type=int, help='Langkah sliding window')
    parser.add_argument('--windowLenth', type=int, default=3, help='Panjang maksimum sliding window')
    parser.add_argument('--batchSize', type=int, default=256, help='Ukuran batch input')
    parser.add_argument('--hiddenSize', type=int, default=100, help='Ukuran hidden state')
    parser.add_argument('--epoch', type=int, default=10, help='Jumlah epoch untuk training')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--lr_dc', type=float, default=0.1, help='Tingkat penurunan learning rate')
    parser.add_argument('--lr_dc_step', type=int, default=3, help='Jumlah langkah sebelum learning rate turun')
    parser.add_argument('--l2', type=float, default=1e-5, help='L2 penalty')
    parser.add_argument('--step', type=int, default=1, help='Jumlah propagasi GNN')
    parser.add_argument('--nonhybrid', action='store_true', help='Hanya gunakan preferensi global untuk prediksi')
    parser.add_argument('--input_json', default=None, type=str, help='Path ke file JSON dengan trial sebelumnya')

    # Parse arguments with default values
    global opt
    opt = parser.parse_args(args=[])
    os.environ["CUDA_VISIBLE_DEVICES"] = opt.cuda_device

    # Set up direktori input dan output
    global INPUT_DIR  # Make INPUT_DIR global too
    INPUT_DIR = "/content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5"
    OUTPUT_DIR = "/content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/output1"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    start_time = time.time()
    print(f"Starting hyperparameter tuning with Optuna at {datetime.datetime.now()}")
    print(f"Using data_size: {opt.data_size}")
    print(f"Input directory: {INPUT_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print("Optimizing for hit@20 on next-new items only")

    # Initialize all_trial_results
    all_trial_results = []

    # Set up hasil direktori
    results_dir = os.path.join(OUTPUT_DIR, "optuna_results")
    os.makedirs(results_dir, exist_ok=True)

    # Cek input JSON di direktori input
    input_json_path = os.path.join(INPUT_DIR, "optuna_results", "all_trials.json")

    # Muat data trial sebelumnya
    if os.path.exists(input_json_path):
        try:
            with open(input_json_path, 'r') as f:
                all_trial_results = json.load(f)
            print(f"Loaded {len(all_trial_results)} existing trials from {input_json_path}")

            # Tampilkan informasi trial yang sudah ada
            print("\nExisting trials:")
            for t in all_trial_results:
                print(f"Trial {t['number']}: lr={t['params']['lr']}, "
                      f"hidden_size={t['params']['hidden_size']}, "
                      f"l2={t['params']['l2']} → hit@20={t['hit@20']:.4f}")
            print()
        except Exception as e:
            print(f"Error loading existing trials: {e}")
            all_trial_results = []
    else:
        print("No existing trials found. Starting fresh.")
        all_trial_results = []

    # Define parameter ranges that will be used in the objective function
    parameter_ranges = {
        'lr': [0.005, 0.001, 0.0005],
        'hidden_size': [60, 80, 100],
        'l2': [0.001, 0.0001, 0.00001]
    }

    print(f"Using random sampling with Optuna to select parameters")
    print(f"Parameter ranges:")
    print(f"  lr: {parameter_ranges['lr']}")
    print(f"  hidden_size: {parameter_ranges['hidden_size']}")
    print(f"  l2: {parameter_ranges['l2']}")

    # Create Optuna study with random sampler
    study = optuna.create_study(
        direction="maximize",
        sampler=RandomSampler(seed=SEED)
    )

    # Total number of trials we want
    n_trials = 8

    # Calculate remaining trials
    remaining_trials = max(0, n_trials - len(all_trial_results))
    print(f"Will run {remaining_trials} more trials to reach total of {n_trials}")

    # Jika semua trial sudah selesai, hanya tampilkan hasil
    if remaining_trials == 0:
        print("All trials already completed. No new trials to run.")

        # Find best trial from existing results
        best_trial = max(all_trial_results, key=lambda x: x['hit@20'])
        print("\nBest Trial Results:")
        print(f"Trial {best_trial['number']}: lr={best_trial['params']['lr']}, "
              f"hidden_size={best_trial['params']['hidden_size']}, "
              f"l2={best_trial['params']['l2']}")
        print(f"Hit@20: {best_trial['hit@20']:.4f}, Hit@10: {best_trial['hit@10']:.4f}")
    else:
        # Setup callback to properly show trial numbers
        last_trial_number = -1
        if all_trial_results:
            last_trial_number = max(t['number'] for t in all_trial_results)

        def trial_callback(study, trial):
            # Callback only fires for successful trials (not pruned)
            adjusted_number = last_trial_number + 1 + len(all_trial_results) - len(all_trial_results_before_optimize)
            print(f"\nRunning Trial {adjusted_number} with parameters:")
            print(f"  lr={trial.params['lr']}")
            print(f"  hidden_size={trial.params['hidden_size']}")
            print(f"  l2={trial.params['l2']}")

        # Store copy of all_trial_results before optimization
        all_trial_results_before_optimize = copy.deepcopy(all_trial_results)

        # Run optimization with pruning handling
        pruned_count = 0
        max_prune_attempts = 50  # Maximum attempts to avoid infinite loop

        try:
            while remaining_trials > 0 and pruned_count < max_prune_attempts:
                # Save starting count to detect if a trial was added
                starting_trial_count = len(all_trial_results)

                try:
                    # Run a single trial
                    study.optimize(
                        objective,
                        n_trials=1,
                        timeout=None,
                        callbacks=[trial_callback],
                        catch=(optuna.exceptions.TrialPruned,)
                    )

                    # Check if a new trial was added (not pruned)
                    if len(all_trial_results) > starting_trial_count:
                        remaining_trials -= 1
                        pruned_count = 0  # Reset pruned counter on success
                    else:
                        # Trial was pruned due to duplicate parameters
                        pruned_count += 1
                        print(f"Pruned trial attempt {pruned_count}/{max_prune_attempts}")

                except Exception as e:
                    if isinstance(e, optuna.exceptions.TrialPruned):
                        # Trial was pruned due to duplicate parameters
                        pruned_count += 1
                        print(f"Pruned trial attempt {pruned_count}/{max_prune_attempts}")
                    else:
                        # Other error - reraise
                        raise e

            if pruned_count >= max_prune_attempts:
                print(f"Stopped after {max_prune_attempts} pruned attempts. All parameter combinations may have been tried.")

            # Print optimization results
            print("\nHyperparameter Optimization Results:")

            # Find best trial from all_trial_results
            if all_trial_results:
                best_trial = max(all_trial_results, key=lambda x: x['hit@20'])
                print(f"Best trial: {best_trial['number']}")
                print(f"Best parameters: lr={best_trial['params']['lr']}, "
                      f"hidden_size={best_trial['params']['hidden_size']}, "
                      f"l2={best_trial['params']['l2']}")
                print(f"Best value (Hit@20 for next-new): {best_trial['hit@20']:.4f}")

        except Exception as e:
            print(f"Error during optimization: {e}")
            import traceback
            traceback.print_exc()

    # Create DataFrame with all trial results
    if all_trial_results:
        trials_df = pd.DataFrame([
            {
                'trial': t['number'],
                'hit@20': t['hit@20'],
                'hit@10': t['hit@10'],
                'lr': t['params']['lr'],
                'hidden_size': t['params']['hidden_size'],
                'l2': t['params']['l2'],
                'datetime': t['datetime']
            }
            for t in all_trial_results
        ])

        # Save final results
        results_path = os.path.join(OUTPUT_DIR, "final_hyperparameter_results.csv")
        trials_df.to_csv(results_path, index=False)
        print(f"\nSaved final results to: {results_path}")

    end_time = time.time()
    print(f"Total tuning time: {(end_time - start_time) / 60:.2f} minutes")

if __name__ == "__main__":
    main()

[I 2025-06-26 22:28:38,498] A new study created in memory with name: no-name-98e5622d-ae7a-4a7e-a533-867950755dbb
[I 2025-06-26 22:28:38,499] Trial 0 pruned. 
[I 2025-06-26 22:28:38,500] Trial 1 pruned. 
[I 2025-06-26 22:28:38,501] Trial 2 pruned. 
[I 2025-06-26 22:28:38,503] Trial 3 pruned. 
[I 2025-06-26 22:28:38,504] Trial 4 pruned. 


Starting hyperparameter tuning with Optuna at 2025-06-26 22:28:38.495153
Using data_size: 0.1
Input directory: /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5
Output directory: /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/output1
Optimizing for hit@20 on next-new items only
Loaded 4 existing trials from /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/optuna_results/all_trials.json

Existing trials:
Trial 0: lr=0.001, hidden_size=100, l2=1e-05 → hit@20=26.2525
Trial 1: lr=0.001, hidden_size=60, l2=0.0001 → hit@20=25.8425
Trial 2: lr=0.0005, hidden_size=60, l2=1e-05 → hit@20=20.9040
Trial 3: lr=0.0005, hidden_size=100, l2=0.0001 → hit@20=25.4934

Using random sampling with Optuna to select parameters
Parameter ranges:
  lr: [0.005, 0.001, 0.0005]
  hidden_size: [60, 80, 100]
  l2: [0.001, 0.0001, 1e-05]
Will run 4 more trials to reach total of 8

Skipping duplicate parameters: lr=0.001, hidden_size=60, l2=0.0001
This combination wa

[I 2025-06-26 23:47:41,080] Trial 5 finished with value: 24.886301485513762 and parameters: {'lr': 0.005, 'hidden_size': 100, 'l2': 0.0001}. Best is trial 5 with value: 24.886301485513762.
[I 2025-06-26 23:47:41,082] Trial 6 pruned. 



Running Trial 5 with parameters:
  lr=0.005
  hidden_size=100
  l2=0.0001

Skipping duplicate parameters: lr=0.001, hidden_size=100, l2=1e-05
This combination was already tried in Trial 0

Running Trial 5 with parameters:
  lr=0.001
  hidden_size=100
  l2=1e-05
Pruned trial attempt 1/50


Trial 5: lr=0.0005, hidden_size=80, l2=0.001
Epoch: 1/10
start training:  2025-06-26 23:47:41.202673
[0/868] Loss: 11.0079
[174/868] Loss: 10.0791
[348/868] Loss: 8.5017
[522/868] Loss: 9.4622
[696/868] Loss: 9.4512
	Loss:	8116.457
start predicting (next-new):  2025-06-26 23:49:49.230026
hit:
[0.83470677 2.04859212 2.91303119 3.78407744 4.56886331 5.22407509
 5.75081397 6.22469543 6.70885472 7.12106919 7.5402579  7.87208504
 8.10774104 8.35257368 8.5625351  8.81213959 9.02467047 9.2217846
 9.41522808 9.61894939]
mrr:
[0.83470677 1.44164945 1.7297958  1.94755736 2.10451454 2.2137165
 2.28896491 2.34820009 2.40199557 2.44321702 2.48132508 2.50897734
 2.52710473 2.54459278 2.5585902  2.57419048 2.586692

[I 2025-06-27 01:04:37,101] Trial 7 finished with value: 21.74752506139169 and parameters: {'lr': 0.0005, 'hidden_size': 80, 'l2': 0.001}. Best is trial 5 with value: 24.886301485513762.



Running Trial 6 with parameters:
  lr=0.0005
  hidden_size=80
  l2=0.001


Trial 6: lr=0.001, hidden_size=80, l2=1e-05
Epoch: 1/10
start training:  2025-06-27 01:04:37.209669
[0/868] Loss: 11.0040
[174/868] Loss: 9.9892
[348/868] Loss: 7.7443
[522/868] Loss: 9.2893
[696/868] Loss: 8.9057
	Loss:	7707.102
start predicting (next-new):  2025-06-27 01:06:45.990468
hit:
[ 2.31214509  4.915006    7.04251719  9.05587103 10.42649331 11.58385059
 12.64393553 13.54911886 14.24030305 14.90762799 15.46446623 15.93541117
 16.36120706 16.76497902 17.11993129 17.45983387 17.74798022 18.0401643
 18.32133641 18.57607981]
mrr:
[2.31214509 3.61357555 4.32274594 4.8260844  5.10020886 5.29310174
 5.44454245 5.55769036 5.6344886  5.7012211  5.75184276 5.79108817
 5.8238417  5.85268255 5.87634604 5.89758995 5.91453973 5.93077218
 5.94557072 5.95830789]
pre:
[2.31214509 2.457503   2.34750573 2.26396776 2.08529866 1.93064176
 1.8062765  1.69363986 1.58225589 1.4907628  1.40586057 1.32795093
 1.25855439 1.19749

[I 2025-06-27 02:22:19,554] Trial 8 finished with value: 25.909679882245413 and parameters: {'lr': 0.001, 'hidden_size': 80, 'l2': 1e-05}. Best is trial 8 with value: 25.909679882245413.
[I 2025-06-27 02:22:19,557] Trial 9 pruned. 
[I 2025-06-27 02:22:19,558] Trial 10 pruned. 



Running Trial 7 with parameters:
  lr=0.001
  hidden_size=80
  l2=1e-05

Skipping duplicate parameters: lr=0.005, hidden_size=100, l2=0.0001
This combination was already tried in Trial 4

Running Trial 7 with parameters:
  lr=0.005
  hidden_size=100
  l2=0.0001
Pruned trial attempt 1/50

Skipping duplicate parameters: lr=0.0005, hidden_size=80, l2=0.001
This combination was already tried in Trial 5

Running Trial 7 with parameters:
  lr=0.0005
  hidden_size=80
  l2=0.001
Pruned trial attempt 2/50


Trial 7: lr=0.0005, hidden_size=100, l2=1e-05
Epoch: 1/10
start training:  2025-06-27 02:22:19.691707
[0/868] Loss: 11.0129
[174/868] Loss: 10.0496
[348/868] Loss: 8.2223
[522/868] Loss: 9.5550
[696/868] Loss: 9.2534
	Loss:	7937.822
start predicting (next-new):  2025-06-27 02:24:28.679907
hit:
[ 1.4341246   3.41627788  5.03320107  6.45778197  7.59495065  8.62383503
  9.43761907 10.10090628 10.66178225 11.21825343 11.63597388 12.07168054
 12.43030345 12.78268626 13.07156675 13.35934604 13.62

[I 2025-06-27 03:40:41,585] Trial 11 finished with value: 25.04964559833499 and parameters: {'lr': 0.0005, 'hidden_size': 100, 'l2': 1e-05}. Best is trial 8 with value: 25.909679882245413.



Running Trial 8 with parameters:
  lr=0.0005
  hidden_size=100
  l2=1e-05

Hyperparameter Optimization Results:
Best trial: 0
Best parameters: lr=0.001, hidden_size=100, l2=1e-05
Best value (Hit@20 for next-new): 26.2525

Saved final results to: /content/drive/MyDrive/TUGAS_AKHIR/dataset/Music4all_new/resullt5/output1/final_hyperparameter_results.csv
Total tuning time: 312.05 minutes
